# Model Training and Evaluation

This notebook uses the reusable project modules to train and evaluate the baseline CNN for MNIST digit recognition.

## 1. Project Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

from src.data.data_loader import get_class_names, get_dataset_summary, load_mnist_data
from src.evaluation.evaluate_model import evaluate_trained_model
from src.models.cnn_model import build_baseline_cnn
from src.preprocessing.image_preprocessor import preprocess_mnist_data
from src.training.train_model import train_baseline_model
from src.utils.paths import MODELS_DIR, REPORTS_DIR

sns.set_theme(style="whitegrid")

## 2. Load MNIST Dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = load_mnist_data()
get_dataset_summary(x_train, y_train, x_test, y_test)

## 3. Preprocess Images

In [ ]:
x_train_processed, y_train_processed, x_test_processed, y_test_processed = preprocess_mnist_data(
    x_train,
    y_train,
    x_test,
    y_test,
)

print(x_train_processed.shape, y_train_processed.shape)
print(x_test_processed.shape, y_test_processed.shape)

## 4. Build CNN Model

In [ ]:
model = build_baseline_cnn()
model.summary()

## 5. Train Model

In [ ]:
# Runs the same training workflow as: python -m src.training.train_model
training_summary = train_baseline_model(epochs=6, batch_size=128)
training_summary

## 6. Evaluate Model

In [ ]:
# Runs the same evaluation workflow as: python -m src.evaluation.evaluate_model
evaluation_summary = evaluate_trained_model()
evaluation_summary

## 7. Confusion Matrix

In [ ]:
confusion_matrix_df = pd.read_csv(REPORTS_DIR / "confusion_matrix.csv", index_col=0)

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix_df, annot=True, fmt="d", cmap="Blues")
plt.title("MNIST Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

## 8. Sample Predictions

In [ ]:
from tensorflow.keras.models import load_model

trained_model = load_model(MODELS_DIR / "mnist_cnn_baseline.keras")
probabilities = trained_model.predict(x_test_processed[:25], verbose=0)
predicted_labels = np.argmax(probabilities, axis=1)

fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for index, ax in enumerate(axes.ravel()):
    ax.imshow(x_test[index], cmap="gray")
    title_color = "green" if y_test[index] == predicted_labels[index] else "red"
    ax.set_title(f"T:{y_test[index]} P:{predicted_labels[index]}", color=title_color)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 9. Key Findings

- Record final training and validation accuracy from `reports/training_summary.json`.
- Record test accuracy and test loss from `reports/evaluation_summary.json`.
- Inspect confusion matrix errors to identify the most commonly confused digits.
- Use saved plots in `images/evaluation/` for reporting and later README updates.